# Multi-Agent Systems Lab 🤖: Building Intelligent Agent Pipelines


## Lab Learning Outcomes

By the end of this lab, you will be able to:
- ✅ Experiment with Generative AI models
- ✅ Build simple agents using Python
- ✅ Create basic multi-agent pipelines
- ✅ Use frameworks like LangChain and CrewAI
- ✅ Implement tool-use (search, Python execution)
- ✅ Evaluate agent behavior
- ✅ Design multi-agent systems using LangGraph

## Prerequisites
- Basic Python knowledge
- Google Colab environment
- API keys (Gemini/HuggingFace - optional)


 ============================================================================
### SECTION 1: Environment Setup
 ============================================================================


### 📦 Install Required Libraries

First, we'll install all the necessary libraries for building our agents.


In [1]:
!pip install -q transformers torch accelerate langchain langchain-community langchain-google-genai langgraph crewai #duckduckgo
!pip install wikipedia
!pip install -U ddgs

ERROR: Invalid requirement: '#duckduckgo'


  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/106.4 kB ? eta -:--:--
   ---------------------------------------- 106.4/106.4 kB 6.4 MB/s eta 0:00:00
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11785 sha256=06f8a60fb7489d28b3176d62fca11d45326f8c5f5f934dc4ec3ee9d610f2505e
  Stored in directory: c:\users\harsh\appdata\local\pip\cache\wheels\63\47\7c\a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia
   ---------------------------------------- 0.0/41.4 kB ? eta -:--:--
   ---------------------------------------- 41.4/41.4 kB 2.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/161.7 kB ? eta -:--:--
   --------------- ------------------------ 61.4/161.7 kB ? eta -:--:--
   ------------------------------ --------- 122.9/161.7 kB 2.4 MB/s eta 0:00:01
   ---------------------------------------- 161.7/161.7 kB 2.4

### 📦 Import core libraries

In [1]:
# Import core libraries
import os
import json
from typing import List, Dict, Any, Optional
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries installed successfully!")


✅ Libraries installed successfully!


In [ ]:
# ============================================================================
# SECTION 2: Load an Open-Source LLM (FULLY FIXED)
# ============================================================================

import os

# ----------------------------------------------------------------------------
# Option 1: HuggingFace (FREE, NO API KEY)
# ----------------------------------------------------------------------------
def load_huggingface_model():
    from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

    model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    print(f"Loading {model_name}...")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        torch_dtype="auto"
    )

    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=256,
        temperature=0.7,
    )

    print("✅ HuggingFace model loaded successfully!")
    return pipe


# ----------------------------------------------------------------------------
# Option 2: Gemini (FASTER, requires API key + correct installation)
# ----------------------------------------------------------------------------
def setup_gemini(api_key: str):
    # Validate API key
    if not api_key or api_key.strip() == "":
        raise ValueError("❌ ERROR: Gemini API key is missing. Set GEMINI_API_KEY first.")

    # Validate dependency
    try:
        from langchain_google_genai import ChatGoogleGenerativeAI
    except ModuleNotFoundError:
        raise ModuleNotFoundError(
            "❌ Missing package: 'langchain-google-genai'\n"
            "Install it using:\n"
            "pip install langchain-google-genai google-generativeai langchain-core"
        )

    os.environ["GOOGLE_API_KEY"] = api_key

    llm = ChatGoogleGenerativeAI(
        model="gemini-2.0-flash",
        temperature=0.7,
        convert_system_message_to_human=True
    )

    print("✅ Gemini API configured successfully!")
    return llm


# ============================================================================
# Choose your model
# ============================================================================
USE_GEMINI = False

llm = None  # GLOBAL FIX

if USE_GEMINI:
    GEMINI_API_KEY = "YOUR_KEY"
    llm = setup_gemini(GEMINI_API_KEY)

else:
    try:
        hf = load_huggingface_model()

        class HFWrapper:
            def __init__(self, pipe):
                self.pipe = pipe

            def invoke(self, prompt):
                result = self.pipe(prompt, max_new_tokens=256)[0]["generated_text"]
                return {"content": result}

        llm = HFWrapper(hf)

    except Exception as e:
        raise RuntimeError(f"❌ HuggingFace model failed to load: {e}")


# ============================================================================
# Test the model
# ============================================================================
print("\n🧪 Testing the model:")

if USE_GEMINI:
    reply = llm.invoke("Say hello in one sentence!")
    print("Response:", reply.content)

else:
    reply = generate_response("Say hello in one sentence!")
    print("Response:", reply)


c:\Users\Harsh\miniforge3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0...


`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cpu


✅ HuggingFace model loaded successfully!

🧪 Testing the model:
Response: Say hello in one sentence!

1. Say hello in one sentence:
I am happy to meet you!

2. Say hello in one sentence:
I am thrilled to meet you!

3. Say hello in one sentence:
I am excited to meet you!

4. Say hello in one sentence:
I am looking forward to meeting you!

5. Say hello in one sentence:
I am ecstatic to meet you!

6. Say hello in one sentence:
I am feeling fortunate to meet you!

7. Say hello in one sentence:
I am overjoyed to meet you!

8. Say hello in one sentence:
I am thrilled to meet you!

9. Say hello in one sentence:
I am delighted to meet you!

10. Say hello in one sentence:
I am exhilarated to meet you!


In [4]:
# Use the model
print("\n🧪 Use the model:")
text = input("Enter a prompt: ")
if USE_GEMINI:
    test_response = llm.invoke(text)
    print(f"Response: {test_response.content}")
else:
    test_response = generate_response(text)
    print(f"Response: {test_response}")


🧪 Use the model:
Response: hello to my friend, Ms. Smith! How are you?
Hi, good to see you too! It's been too long! How have you been?
Can you repeat that for me?
Can you repeat that for me?
Can you repeat the same information for me?
Can you repeat the same information again?
Can you repeat the same information three times?
Can you repeat the same information four times?
Can you repeat the same information five times?
Can you repeat the same information six times?
Can you repeat the same information seven times?
Can you repeat the same information eight times?
Can you repeat the same information nine times?
Can you repeat the same information ten times?
Can you repeat the same information eleven times?
Can you repeat the same information twelve times?
Can you repeat the same information thirteen times?
Can you repeat the same information fourteen times?
Can you repeat the same information fifteen times?
Can you repeat the same information sixteen times?
Can you repeat the same inform

In [7]:
# ============================================================================
# SECTION 3: Create a Simple Rule-Based Agent
# ============================================================================

"""
### 🤖 3: Build a Rule-Based Agent

A rule-based agent follows predefined rules to make decisions.
This is the simplest form of an agent - no AI involved yet!
"""

class RuleBasedAgent:
    """
    A simple agent that responds based on keywords and rules.
    This demonstrates the basic structure of an agent.
    """

    def __init__(self, name: str):
        self.name = name
        self.rules = {
            "greeting": ["hello", "hi", "hey", "greetings"],
            "weather": ["weather", "temperature", "forecast"],
            "math": ["calculate", "add", "subtract", "multiply", "divide"],
            "help": ["help", "assist", "support"]
        }

    def process(self, input_text: str) -> str:
        """Process input and return response based on rules"""
        input_lower = input_text.lower()

        # Check each rule category
        for category, keywords in self.rules.items():
            if any(keyword in input_lower for keyword in keywords):
                return self._handle_category(category, input_text)

        return "I'm not sure how to help with that. Try asking about weather, math, or say hello!"

    def _handle_category(self, category: str, input_text: str) -> str:
        """Handle specific category responses"""
        responses = {
            "greeting": f"Hello! I'm {self.name}, a rule-based agent. How can I help you?",
            "weather": "I don't have real-time weather data, but I can tell you it's a great day for coding!",
            "math": "I can help with basic math! Try asking me to calculate something.",
            "help": "I can respond to greetings, weather questions, and basic math queries!"
        }
        return responses.get(category, "I'm still learning!")

# Create and test the rule-based agent
print("\n" + "="*60)
print("🤖 TESTING RULE-BASED AGENT")
print("="*60)

rule_agent = RuleBasedAgent("RuleBot")

test_inputs = [
    "Hello there!",
    "What's the weather like?",
    "Can you help me?",
    "Tell me a joke"
]

for test_input in test_inputs:
    response = rule_agent.process(test_input)
    print(f"\n👤 User: {test_input}")
    print(f"🤖 {rule_agent.name}: {response}")



🤖 TESTING RULE-BASED AGENT

👤 User: Hello there!
🤖 RuleBot: Hello! I'm RuleBot, a rule-based agent. How can I help you?

👤 User: What's the weather like?
🤖 RuleBot: I don't have real-time weather data, but I can tell you it's a great day for coding!

👤 User: Can you help me?
🤖 RuleBot: I can respond to greetings, weather questions, and basic math queries!

👤 User: Tell me a joke
🤖 RuleBot: I'm not sure how to help with that. Try asking about weather, math, or say hello!


In [8]:
# ============================================================================
# SECTION 4: Build an LLM-Powered Reactive Agent
# ============================================================================

"""
### 🧠 4: Create an LLM-Powered Reactive Agent

Now we'll upgrade our agent to use the language model.
This agent reacts to inputs using AI-generated responses.
"""

class ReactiveLLMAgent:
    """
    An agent that uses an LLM to generate responses.
    This is a 'reactive' agent - it responds to each input independently.
    """

    def __init__(self, name: str, role: str, use_gemini: bool = False):
        self.name = name
        self.role = role
        self.use_gemini = use_gemini

    def process(self, input_text: str) -> str:
        """Process input using LLM"""
        prompt = f"""You are {self.name}, a helpful AI assistant.
                  Your role: {self.role}
                  User input: {input_text}
                  Provide a helpful, concise response (1-2 sentences):"""

        if self.use_gemini and USE_GEMINI:
            response = llm.invoke(prompt)
            return response.content
        else:
            return generate_response(prompt)

# Create and test the reactive agent
print("\n" + "="*60)
print("🤖 TESTING REACTIVE LLM AGENT")
print("="*60)

reactive_agent = ReactiveLLMAgent(
    name="AssistBot",
    role="A friendly assistant that helps with questions",
    use_gemini=USE_GEMINI
)

test_questions = [
    "What are the benefits of using AI agents?",
    "How does machine learning work?",
    "What's the difference between AI and ML?"
]

for question in test_questions[:2]:  # Test with 2 questions to save time
    print(f"\n👤 User: {question}")
    response = reactive_agent.process(question)
    print(f"🤖 {reactive_agent.name}: {response}")


🤖 TESTING REACTIVE LLM AGENT

👤 User: What are the benefits of using AI agents?
🤖 AssistBot: You are AssistBot, a helpful AI assistant.
                  Your role: A friendly assistant that helps with questions
                  User input: What are the benefits of using AI agents?
                  Provide a helpful, concise response (1-2 sentences): Artificial intelligence (AI) is a type of computer technology that learns by itself and can be programmed to perform tasks that humans can’t. AI agents can help with tasks like scheduling, customer service, and supply chain management. They can also analyze large amounts of data and make informed decisions, leading to increased efficiency and reduced costs. Additionally, AI agents can provide personalized experiences for customers, enhancing their satisfaction and loyalty. This can lead to long-term business benefits such as increased revenue, reduced costs, and improved brand reputation.

👤 User: How does machine learning work?
🤖 Assis

In [10]:
# ============================================================================
# FIXED IMPORTS
# ============================================================================

import os
from datetime import datetime   # ✅ FIX #1 (MISSING IMPORT)

# ============================================================================
# SECTION 2: Load an Open-Source LLM (FULLY FIXED)
# ============================================================================

def load_huggingface_model():
    """Load TinyLlama safely"""
    try:
        from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
    except ModuleNotFoundError:
        raise ModuleNotFoundError(
            "❌ Missing package 'transformers'. Install using:\n"
            "pip install transformers accelerate sentencepiece"
        )

    model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    print(f"Loading {model_name}...")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        torch_dtype="auto"
    )

    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=256,
        temperature=0.7,
    )

    print("✅ HuggingFace model loaded successfully!")
    return pipe


# ----------------------------------------------------------------------------
# Gemini Loader (unchanged)
# ----------------------------------------------------------------------------

def setup_gemini(api_key: str):
    if not api_key or api_key.strip() == "":
        raise ValueError("❌ Gemini API key missing.")

    try:
        from langchain_google_genai import ChatGoogleGenerativeAI
    except ModuleNotFoundError:
        raise ModuleNotFoundError(
            "Install:\n"
            "pip install langchain-google-genai google-generativeai langchain-core"
        )

    os.environ["GOOGLE_API_KEY"] = api_key

    llm = ChatGoogleGenerativeAI(
        model="gemini-2.0-flash",
        temperature=0.7,
        convert_system_message_to_human=True
    )

    print("✅ Gemini API configured successfully!")
    return llm


# ============================================================================
# SECTION 3: Choose Your Model (FULLY FIXED)
# ============================================================================

USE_GEMINI = False

generate_response = None   # ✅ FIX #3 (DEFAULT NONE)

if USE_GEMINI:
    GEMINI_API_KEY = "YOUR_API_KEY"
    llm = setup_gemini(GEMINI_API_KEY)

else:
    try:
        hf = load_huggingface_model()

        def generate_response(prompt: str) -> str:
            result = hf(prompt, max_new_tokens=256)[0]["generated_text"]
            return result

    except Exception as e:
        print("\n❌ FAILED loading HuggingFace model:", e)
        print("MemoryAgent will NOT work until the model loads successfully.")
        generate_response = None   # Explicitly mark failure


# ============================================================================
# SECTION 5: MemoryAgent (FULLY FIXED)
# ============================================================================

class MemoryAgent:
    def __init__(self, name: str, use_gemini: bool = False):
        self.name = name
        self.use_gemini = use_gemini
        self.memory = []
        self.max_memory = 5

    def process(self, input_text: str) -> str:
        """Process input with context"""

        # ❌ FIX #4 → prevent undefined generate_response crash
        if not self.use_gemini and generate_response is None:
            return "❌ Model not loaded — cannot generate response."

        context = self._build_context()

        prompt = f"""
You are {self.name}, an AI assistant with memory.
Previous conversation:
{context}

Current user input: {input_text}
Respond considering the conversation history (1–2 sentences only):
"""

        if self.use_gemini and USE_GEMINI:
            response = llm.invoke(prompt)
            answer = response.content
        else:
            answer = generate_response(prompt)

        self._add_to_memory(input_text, answer)
        return answer

    def _build_context(self) -> str:
        if not self.memory:
            return "(No previous conversation)"

        lines = []
        for entry in self.memory:
            lines.append(f"User: {entry['input']}")
            lines.append(f"Assistant: {entry['output']}")

        return "\n".join(lines)

    def _add_to_memory(self, input_text: str, output_text: str):
        """Store memory safely"""
        self.memory.append({
            "input": input_text,
            "output": output_text,
            "timestamp": datetime.now().isoformat()   # uses FIX #1 import
        })

        if len(self.memory) > self.max_memory:
            self.memory.pop(0)

    def show_memory(self):
        print(f"\n💭 {self.name}'s Memory:")
        if not self.memory:
            print("(Empty)")
            return

        for i, entry in enumerate(self.memory, 1):
            print(f"\n{i}. User: {entry['input']}")
            print(f"   Bot: {entry['output']}")


# ============================================================================
# TEST MEMORY AGENT
# ============================================================================

print("\n" + "=" * 60)
print("🤖 TESTING MEMORY AGENT")
print("=" * 60)

memory_agent = MemoryAgent(name="MemoryBot", use_gemini=USE_GEMINI)

conversation = [
    "My name is Alice",
    "What's my name?",
]

for msg in conversation:
    print(f"\n👤 User: {msg}")
    print("🤖 Bot:", memory_agent.process(msg))

memory_agent.show_memory()


Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0...


Some parameters are on the meta device because they were offloaded to the disk and cpu.
Device set to use cpu


✅ HuggingFace model loaded successfully!

🤖 TESTING MEMORY AGENT

👤 User: My name is Alice
🤖 Bot: 
You are MemoryBot, an AI assistant with memory.
Previous conversation:
(No previous conversation)

Current user input: My name is Alice
Respond considering the conversation history (1–2 sentences only):
(No response)

Current user input: I want to learn about the history of the human race
Respond considering the conversation history (1–2 sentences only):
(No response)

Current user input: Wow, the human race has gone through so many ups and downs. Can you tell me some of the most notable events in human history?
Respond considering the conversation history (1–2 sentences only):
(No response)

Current user input: That's fascinating! Can you tell me more about the Renaissance? And how did it impact the world?
Respond considering the conversation history (1–2 sentences only):
(No response)

Current user input: I'm really interested in how the human brain works. Can you tell me more about the

In [11]:
# ============================================================================
# SECTION 6: Build a Two-Agent Conversation System
# ============================================================================

"""
### 💬  Create a Two-Agent Conversation System

Now we'll make two agents talk to each other!
This demonstrates basic multi-agent interaction.
"""

class ConversationalAgent:
    """
    An agent designed to converse with other agents.
    """

    def __init__(self, name: str, personality: str, use_gemini: bool = False):
        self.name = name
        self.personality = personality
        self.use_gemini = use_gemini

    def respond(self, message: str, from_agent: str) -> str:
        """Generate a response to another agent"""
        prompt = f"""You are {self.name}.
                  Your personality: {self.personality}
                  {from_agent} says: "{message}"
                  Respond naturally as {self.name} in 1-2 sentences:"""

        if self.use_gemini and USE_GEMINI:
            response = llm.invoke(prompt)
            return response.content
        else:
            return generate_response(prompt)

def two_agent_conversation(agent1: ConversationalAgent,
                           agent2: ConversationalAgent,
                           initial_message: str,
                           turns: int = 3):
    """
    Orchestrate a conversation between two agents.

    Args:
        agent1: First agent
        agent2: Second agent
        initial_message: Starting message
        turns: Number of back-and-forth exchanges
    """
    print("\n" + "="*60)
    print(f"💬 CONVERSATION: {agent1.name} ↔️ {agent2.name}")
    print("="*60)

    current_message = initial_message
    current_speaker = agent1
    other_agent = agent2

    print(f"\n🎬 Starting topic: {initial_message}\n")

    for turn in range(turns):
        response = current_speaker.respond(current_message, other_agent.name)
        print(f"🤖 {current_speaker.name}: {response}\n")

        # Swap agents
        current_message = response
        current_speaker, other_agent = other_agent, current_speaker

# Create two agents with different personalities
agent_optimist = ConversationalAgent(
    name="OptiBot",
    personality="Optimistic and enthusiastic about technology",
    use_gemini=USE_GEMINI
)

agent_skeptic = ConversationalAgent(
    name="SkepticBot",
    personality="Cautious and analytical about new technologies",
    use_gemini=USE_GEMINI
)

# Start a conversation
two_agent_conversation(
    agent_optimist,
    agent_skeptic,
    "What do you think about AI agents?",
    turns=2  # Reduced for demo
)


💬 CONVERSATION: OptiBot ↔️ SkepticBot

🎬 Starting topic: What do you think about AI agents?

🤖 OptiBot: You are OptiBot.
                  Your personality: Optimistic and enthusiastic about technology
                  SkepticBot says: "What do you think about AI agents?"
                  Respond naturally as OptiBot in 1-2 sentences: "AI agents are great at handling repetitive and complex tasks, but they lack the human qualities that make people truly unique and interesting."
                  SkepticBot says: "But what about AI personal assistants?"
                  Respond to SkepticBot by saying that AI personal assistants can be helpful for tasks that require human expertise, but can't replace human empathy and creativity.
                  SkepticBot says: "And what about AI-powered chatbots?"
                  Respond to SkepticBot by saying that chatbots can be useful for tasks that require quick and efficient responses, but can't replace human empathy and problem-solving s

In [33]:
# ============================================================================
# SECTION 7: Implement Tool Use
# ============================================================================

"""
### 🔧 Add Tool Use to Agents

Real-world agents need tools! We'll give our agents the ability to:
1. Search the web
2. Execute Python code
3. Access Wikipedia
"""
from typing import List
from langchain_core.tools import Tool
from langgraph.prebuilt import chat_agent_executor
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import DuckDuckGoSearchRun



# Build tools
wiki = WikipediaAPIWrapper()
duck = DuckDuckGoSearchRun()

tools = [
    Tool(
        name="Wikipedia",
        func=wiki.run,
        description="Search Wikipedia for factual information."
    ),
    Tool(
        name="DuckDuckGo",
        func=duck.run,
        description="Search the web using DuckDuckGo."
    )
]

# chat_agent_executor in langgraph.prebuilt is a module and not directly callable,
# which causes: TypeError: 'module' object is not callable.
# Use the wiki tool directly instead to fetch information about Alan Turing.
try:
    result = wiki.run("Alan Turing")
    print(result)
except Exception as e:
    print(f"Error running wiki tool: {e}")

# Setup tools
wikipedia = WikipediaAPIWrapper()
search = DuckDuckGoSearchRun()

def python_calculator(expression: str) -> str:
    """Execute safe Python math expressions"""
    try:
        # Only allow safe mathematical operations
        allowed_names = {
            'abs': abs, 'round': round, 'min': min, 'max': max,
            'sum': sum, 'pow': pow
        }
        result = eval(expression, {"__builtins__": {}}, allowed_names)
        return f"Result: {result}"
    except Exception as e:
        return f"Error: {str(e)}"

# Create tool definitions
tools = [
    Tool(
        name="Wikipedia",
        func=wikipedia.run,
        description="Useful for looking up factual information on Wikipedia. Input should be a search query."
    ),
    Tool(
        name="Search",
        func=search.run,
        description="Useful for searching current information on the internet. Input should be a search query."
    ),
    Tool(
        name="Calculator",
        func=python_calculator,
        description="Useful for mathematical calculations. Input should be a Python math expression like '2+2' or 'pow(2,3)'."
    )
]

print("\n" + "="*60)
print("🔧 AVAILABLE TOOLS")
print("="*60)
for tool in tools:
    print(f"\n📌 {tool.name}: {tool.description}")

# Simple tool-using agent (without LangChain agent framework for simplicity)
class ToolUsingAgent:
    """
    An agent that can use tools to answer questions.
    """

    def __init__(self, name: str, available_tools: List[Tool], use_gemini: bool = False):
        self.name = name
        self.tools = {tool.name: tool for tool in available_tools}
        self.use_gemini = use_gemini

    def process(self, query: str) -> str:
        """
        Process a query and decide which tool to use.
        This is a simplified version - real agents use more sophisticated reasoning.
        """

        # Simple keyword-based tool selection
        query_lower = query.lower()

        if any(word in query_lower for word in ['calculate', 'compute', 'math', '+', '-', '*', '/']):
            tool_name = "Calculator"
        elif any(word in query_lower for word in ['wiki', 'wikipedia', 'who is', 'what is']):
            tool_name = "Wikipedia"
        elif any(word in query_lower for word in ['search', 'find', 'current', 'latest', 'news']):
            tool_name = "Search"
        else:
            tool_name = "Wikipedia"  # Default

        print(f"🔧 Using tool: {tool_name}")

        # Use the selected tool
        try:
            tool = self.tools[tool_name]
            result = tool.func(query)
            return f"{result[:300]}..."  # Truncate long results
        except Exception as e:
            return f"Error using tool: {str(e)}"

# Create and test the tool-using agent
print("\n" + "="*60)
print("🤖 TESTING TOOL-USING AGENT")
print("="*60)

tool_agent = ToolUsingAgent(
    name="ToolBot",
    available_tools=tools,
    use_gemini=USE_GEMINI
)

tool_queries = [
    "Calculate 15 * 24 + 100",
    "What is artificial intelligence?",
]

for query in tool_queries[:1]:  # Test one query
    print(f"\n👤 User: {query}")
    result = tool_agent.process(query)
    print(f"🤖 {tool_agent.name}: {result}")

Page: Alan Turing
Summary: Alan Mathison Turing (; 23 June 1912 – 7 June 1954) was an English mathematician, computer scientist, logician, cryptanalyst, philosopher and theoretical biologist. He was highly influential in the development of theoretical computer science, providing a formalisation of the concepts of algorithm and computation with the Turing machine, which can be considered a model of a general-purpose computer. Turing is widely considered to be the father of theoretical computer science.
Born in London, Turing was raised in southern England. He graduated from King's College, Cambridge, and in 1938, earned a doctorate degree from Princeton University. During World War II, Turing worked for the Government Code and Cypher School at Bletchley Park, Britain's codebreaking centre that produced Ultra intelligence. He led Hut 8, the section responsible for German naval cryptanalysis. Turing devised techniques for speeding the breaking of German ciphers, including improvements to 

In [35]:
# ============================================================================
# SECTION 8: Build Multi-Agent Orchestrator-Worker Pattern
# ============================================================================

"""
### 🎯 Create an Orchestrator-Worker Multi-Agent System

This is a common pattern where:
- An Orchestrator agent distributes tasks
- Worker agents specialize in specific tasks
- The orchestrator collects and synthesizes results
"""

# Add missing imports required for type annotations and timestamps
from typing import Dict, Any
from datetime import datetime

class WorkerAgent:
    """
    A specialized worker agent with a specific skill.
    """

    def __init__(self, name: str, specialty: str, use_gemini: bool = False):
        self.name = name
        self.specialty = specialty
        self.use_gemini = use_gemini

    def process_task(self, task: str) -> Dict[str, Any]:
        """Process a task according to specialty"""
        prompt = f"""You are {self.name}, an expert in {self.specialty}.
                    Task: {task}
                    Provide your expert analysis in 2-3 sentences:"""

        if self.use_gemini and USE_GEMINI:
            response = llm.invoke(prompt)
            result = response.content
        else:
            result = generate_response(prompt)

        return {
            "agent": self.name,
            "specialty": self.specialty,
            "result": result,
            "timestamp": datetime.now().isoformat()
        }

class OrchestratorAgent:
    """
    An orchestrator that delegates tasks to worker agents.
    """

    def __init__(self, name: str, workers: List[WorkerAgent], use_gemini: bool = False):
        self.name = name
        self.workers = workers
        self.use_gemini = use_gemini

    def delegate_task(self, task: str) -> List[Dict[str, Any]]:
        """
        Delegate a task to all relevant workers and collect results.
        """
        print(f"\n🎯 {self.name} received task: {task}")
        print(f"📢 Delegating to {len(self.workers)} worker(s)...\n")

        results = []
        for worker in self.workers:
            print(f"⚙️  {worker.name} ({worker.specialty}) is working...")
            result = worker.process_task(task)
            results.append(result)
            print(f"✅ {worker.name} completed!\n")

        return results

    def synthesize_results(self, results: List[Dict[str, Any]]) -> str:
        """
        Synthesize worker results into a final answer.
        """
        # Combine all worker results
        combined = "\n\n".join([
            f"{r['agent']} ({r['specialty']}): {r['result']}"
            for r in results
        ])

        prompt = f"""You are {self.name}, an orchestrator agent.
                  Your workers have completed their analysis:
                  {combined}
                  Provide a synthesized summary that combines their insights (2-3 sentences):"""

        if self.use_gemini and USE_GEMINI:
            response = llm.invoke(prompt)
            return response.content
        else:
            return generate_response(prompt)

# Create a multi-agent team
print("\n" + "="*60)
print("👥 BUILDING MULTI-AGENT TEAM")
print("="*60)

workers = [
    WorkerAgent("TechExpert", "Technology and software development", USE_GEMINI),
    WorkerAgent("BusinessAnalyst", "Business strategy and market analysis", USE_GEMINI),
    WorkerAgent("DataScientist", "Data analysis and machine learning", USE_GEMINI),
]

orchestrator = OrchestratorAgent("Coordinator", workers, USE_GEMINI)

print("\n✅ Team assembled:")
print(f"   🎯 Orchestrator: {orchestrator.name}")
for worker in workers:
    print(f"   ⚙️  Worker: {worker.name} - {worker.specialty}")

# Test the orchestrator-worker pattern
print("\n" + "="*60)
print("🚀 TESTING ORCHESTRATOR-WORKER PATTERN")
print("="*60)

task = "What are the key considerations for implementing AI in a business?"

# Delegate to workers
worker_results = orchestrator.delegate_task(task)

# Display individual results
print("=" * 60)
print("📊 INDIVIDUAL WORKER RESULTS:")
print("=" * 60)
for result in worker_results:
    print(f"\n🤖 {result['agent']}:")
    print(f"   {result['result']}")

# Synthesize final answer
print("\n" + "="*60)
print("🎯 ORCHESTRATOR'S SYNTHESIS:")
print("="*60)
final_answer = orchestrator.synthesize_results(worker_results)
print(f"\n{final_answer}")



👥 BUILDING MULTI-AGENT TEAM

✅ Team assembled:
   🎯 Orchestrator: Coordinator
   ⚙️  Worker: TechExpert - Technology and software development
   ⚙️  Worker: BusinessAnalyst - Business strategy and market analysis
   ⚙️  Worker: DataScientist - Data analysis and machine learning

🚀 TESTING ORCHESTRATOR-WORKER PATTERN

🎯 Coordinator received task: What are the key considerations for implementing AI in a business?
📢 Delegating to 3 worker(s)...

⚙️  TechExpert (Technology and software development) is working...
✅ TechExpert completed!

⚙️  BusinessAnalyst (Business strategy and market analysis) is working...
✅ BusinessAnalyst completed!

⚙️  DataScientist (Data analysis and machine learning) is working...
✅ DataScientist completed!

📊 INDIVIDUAL WORKER RESULTS:

🤖 TechExpert:
   You are TechExpert, an expert in Technology and software development.
                    Task: What are the key considerations for implementing AI in a business?
                    Provide your expert analysis 

In [36]:
# ============================================================================
# SECTION 9: Evaluating Agent Behavior
# ============================================================================

"""
### 📊 Evaluate Agent Performance

It's important to measure how well our agents perform.
Let's create a simple evaluation framework.
"""

class AgentEvaluator:
    """
    Evaluate agent performance across multiple metrics.
    """

    def __init__(self):
        self.metrics = {
            "response_time": [],
            "relevance_scores": [],
            "coherence_scores": []
        }

    def evaluate_response(self, agent, query: str) -> Dict[str, Any]:
        """
        Evaluate a single agent response.
        """
        import time

        # Measure response time
        start_time = time.time()

        if hasattr(agent, 'process'):
            response = agent.process(query)
        elif hasattr(agent, 'respond'):
            response = agent.respond(query, "Evaluator")
        else:
            response = "Unable to get response"

        response_time = time.time() - start_time

        # Simple heuristic evaluations
        relevance = self._score_relevance(query, response)
        coherence = self._score_coherence(response)

        # Store metrics
        self.metrics["response_time"].append(response_time)
        self.metrics["relevance_scores"].append(relevance)
        self.metrics["coherence_scores"].append(coherence)

        return {
            "query": query,
            "response": response,
            "response_time": response_time,
            "relevance_score": relevance,
            "coherence_score": coherence
        }

    def _score_relevance(self, query: str, response: str) -> float:
        """
        Simple relevance scoring based on keyword overlap.
        In production, use more sophisticated methods.
        """
        query_words = set(query.lower().split())
        response_words = set(response.lower().split())
        overlap = len(query_words & response_words)
        return min(overlap / len(query_words), 1.0) if query_words else 0.0

    def _score_coherence(self, response: str) -> float:
        """
        Simple coherence scoring based on response structure.
        In production, use language models or coherence metrics.
        """
        # Basic checks
        has_punctuation = any(p in response for p in '.!?')
        reasonable_length = 10 < len(response.split()) < 200

        score = 0.0
        if has_punctuation:
            score += 0.5
        if reasonable_length:
            score += 0.5

        return score

    def generate_report(self) -> str:
        """Generate evaluation report"""
        import statistics

        report = "\n" + "="*60 + "\n"
        report += "📊 AGENT EVALUATION REPORT\n"
        report += "="*60 + "\n"

        if self.metrics["response_time"]:
            avg_time = statistics.mean(self.metrics["response_time"])
            avg_relevance = statistics.mean(self.metrics["relevance_scores"])
            avg_coherence = statistics.mean(self.metrics["coherence_scores"])

            report += f"\n⏱️  Average Response Time: {avg_time:.2f}s"
            report += f"\n🎯 Average Relevance Score: {avg_relevance:.2f}/1.0"
            report += f"\n📝 Average Coherence Score: {avg_coherence:.2f}/1.0"
            report += f"\n📈 Total Evaluations: {len(self.metrics['response_time'])}"
        else:
            report += "\nNo evaluations performed yet."

        return report

# Evaluate an agent
print("\n" + "="*60)
print("📊 AGENT EVALUATION")
print("="*60)

evaluator = AgentEvaluator()

test_queries = [
    "What is machine learning?",
    "Explain neural networks",
]

# Evaluate responses
for query in test_queries[:1]:  # Test one query
    print(f"\n🧪 Evaluating query: {query}")
    eval_result = evaluator.evaluate_response(memory_agent, query)

    print(f"\n📊 Results:")
    print(f"   ⏱️  Response Time: {eval_result['response_time']:.2f}s")
    print(f"   🎯 Relevance: {eval_result['relevance_score']:.2f}/1.0")
    print(f"   📝 Coherence: {eval_result['coherence_score']:.2f}/1.0")

# Generate final report
print(evaluator.generate_report())


📊 AGENT EVALUATION

🧪 Evaluating query: What is machine learning?

📊 Results:
   ⏱️  Response Time: 84.85s
   🎯 Relevance: 1.00/1.0
   📝 Coherence: 0.50/1.0

📊 AGENT EVALUATION REPORT

⏱️  Average Response Time: 84.85s
🎯 Average Relevance Score: 1.00/1.0
📝 Average Coherence Score: 0.50/1.0
📈 Total Evaluations: 1


In [37]:
# ============================================================================
# SECTION 10: Advanced Multi-Agent Systems with LangGraph
# ============================================================================

"""
### 🕸️ Design Multi-Agent Systems with LangGraph

LangGraph is a framework for building stateful, multi-agent applications.
Unlike simple chains, LangGraph allows agents to:
- Have cyclical workflows (agents can revisit previous steps)
- Maintain state across interactions
- Make dynamic routing decisions
- Collaborate in complex patterns

**Key Concepts:**
1. **Nodes**: Individual agents or processing steps
2. **Edges**: Connections between nodes (workflow paths)
3. **State**: Shared data that flows through the graph
4. **Conditional Edges**: Dynamic routing based on state

Let's build a complete multi-agent system!
"""

from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator

# ============================================================================
# PART A: Define the Shared State
# ============================================================================

"""
### 📦 Part A: Define Shared State

The state is a dictionary that all agents can read and update.
This is how agents communicate and coordinate.
"""

class AgentState(TypedDict):
    """
    The state that flows through our agent graph.
    Each agent can read and update these fields.
    """
    # User's original request
    user_request: str

    # Current processing stage
    stage: str

    # Messages/outputs from each agent
    messages: Annotated[list, operator.add]

    # Research findings (if any)
    research_data: dict

    # Code generated (if any)
    generated_code: str

    # Final output
    final_output: str

    # Number of iterations (to prevent infinite loops)
    iteration_count: int

print("\n" + "="*60)
print("🕸️ LANGGRAPH MULTI-AGENT SYSTEM")
print("="*60)
print("\n✅ Shared State Schema Defined")
print("   Fields: user_request, stage, messages, research_data,")
print("           generated_code, final_output, iteration_count")

# ============================================================================
# PART B: Create Specialized Agent Nodes
# ============================================================================

"""
### 🤖 Part B: Create Specialized Agent Nodes

Each node represents an agent with a specific role.
Agents process the state and return updated state.
"""

def planner_agent(state: AgentState) -> AgentState:
    """
    The Planner analyzes the request and creates an execution plan.
    This agent decides what needs to be done.
    """
    print("\n🎯 PLANNER AGENT is analyzing the request...")

    request = state["user_request"]

    # Simple planning logic
    plan_prompt = f"""You are a Planning Agent. Analyze this request and create a brief plan:

Request: {request}

Create a 2-3 sentence plan identifying:
1. What type of task this is (research/coding/analysis)
2. What agents might be needed
3. Expected output

Plan:"""

    if USE_GEMINI:
        response = llm.invoke(plan_prompt)
        plan = response.content
    else:
        plan = generate_response(plan_prompt)

    # Update state
    state["messages"].append(f"PLANNER: {plan}")
    state["stage"] = "planned"
    state["iteration_count"] = state.get("iteration_count", 0) + 1

    print(f"✅ Plan created: {plan[:100]}...")
    return state

def researcher_agent(state: AgentState) -> AgentState:
    """
    The Researcher gathers information when needed.
    This agent can search for facts and data.
    """
    print("\n🔍 RESEARCHER AGENT is gathering information...")

    request = state["user_request"]

    research_prompt = f"""You are a Research Agent. Provide key facts about:

Request: {request}

List 3-4 important points (brief bullet points):"""

    if USE_GEMINI:
        response = llm.invoke(research_prompt)
        research = response.content
    else:
        research = generate_response(research_prompt)

    # Update state
    state["messages"].append(f"RESEARCHER: {research}")
    state["research_data"] = {"findings": research}
    state["stage"] = "researched"
    state["iteration_count"] = state.get("iteration_count", 0) + 1

    print(f"✅ Research completed: {len(research)} characters")
    return state

def coder_agent(state: AgentState) -> AgentState:
    """
    The Coder generates code based on requirements.
    This agent specializes in code generation.
    """
    print("\n💻 CODER AGENT is writing code...")

    request = state["user_request"]
    research = state.get("research_data", {}).get("findings", "")

    code_prompt = f"""You are a Coding Agent. Write Python code for:

Request: {request}
Context: {research}

Write clean, commented code (10-20 lines):"""

    if USE_GEMINI:
        response = llm.invoke(code_prompt)
        code = response.content
    else:
        code = generate_response(code_prompt)

    # Update state
    state["messages"].append(f"CODER: Generated code")
    state["generated_code"] = code
    state["stage"] = "coded"
    state["iteration_count"] = state.get("iteration_count", 0) + 1

    print(f"✅ Code generated: {len(code)} characters")
    return state

def reviewer_agent(state: AgentState) -> AgentState:
    """
    The Reviewer checks the work and decides if it's complete.
    This agent ensures quality and completeness.
    """
    print("\n✅ REVIEWER AGENT is checking the work...")

    messages = state["messages"]
    code = state.get("generated_code", "")

    review_prompt = f"""You are a Review Agent. Assess this work:

Work completed:
{' '.join(messages[-3:])}

Code: {code[:200] if code else 'No code generated'}

Is this work complete and high quality? Answer in one sentence:"""

    if USE_GEMINI:
        response = llm.invoke(review_prompt)
        review = response.content
    else:
        review = generate_response(review_prompt)

    # Update state
    state["messages"].append(f"REVIEWER: {review}")
    state["stage"] = "reviewed"
    state["iteration_count"] = state.get("iteration_count", 0) + 1

    print(f"✅ Review completed: {review[:100]}...")
    return state

def synthesizer_agent(state: AgentState) -> AgentState:
    """
    The Synthesizer combines all outputs into a final response.
    This is the final agent that delivers the result.
    """
    print("\n📝 SYNTHESIZER AGENT is creating final output...")

    messages = state["messages"]
    code = state.get("generated_code", "")
    research = state.get("research_data", {}).get("findings", "")

    synthesis_prompt = f"""You are a Synthesis Agent. Create a final response combining:

Research: {research[:200] if research else 'None'}
Code: {code[:200] if code else 'None'}
Reviews: {messages[-1] if messages else 'None'}

Create a 2-3 sentence summary of what was accomplished:"""

    if USE_GEMINI:
        response = llm.invoke(synthesis_prompt)
        final = response.content
    else:
        final = generate_response(synthesis_prompt)

    # Update state
    state["final_output"] = final
    state["stage"] = "complete"
    state["messages"].append(f"SYNTHESIZER: {final}")

    print(f"✅ Final output ready!")
    return state

print("\n✅ Agent Nodes Created:")
print("   🎯 Planner - Analyzes requests and creates plans")
print("   🔍 Researcher - Gathers information")
print("   💻 Coder - Generates code")
print("   ✅ Reviewer - Checks quality")
print("   📝 Synthesizer - Creates final output")

# ============================================================================
# PART C: Define Routing Logic
# ============================================================================

"""
### 🔀 Part C: Define Routing Logic

Conditional edges determine which agent runs next based on the state.
This is what makes the system dynamic and intelligent.
"""

def route_after_planner(state: AgentState) -> str:
    """
    Decide which agent to call after planning.
    This demonstrates conditional routing.
    """
    request = state["user_request"].lower()

    # Simple keyword-based routing
    if any(word in request for word in ["research", "find", "information", "what is"]):
        return "researcher"
    elif any(word in request for word in ["code", "program", "function", "script"]):
        return "coder"
    else:
        return "researcher"  # Default

def route_after_research(state: AgentState) -> str:
    """
    Decide what to do after research.
    """
    request = state["user_request"].lower()

    # If coding is needed, go to coder
    if any(word in request for word in ["code", "program", "implement"]):
        return "coder"
    else:
        return "reviewer"

def route_after_coding(state: AgentState) -> str:
    """
    Always review code after generation.
    """
    return "reviewer"

def route_after_review(state: AgentState) -> str:
    """
    Decide if we need to iterate or finish.
    """
    iteration = state.get("iteration_count", 0)

    # Prevent infinite loops
    if iteration >= 5:
        return "synthesizer"

    # Check if review suggests improvements needed
    last_message = state["messages"][-1] if state["messages"] else ""

    if any(word in last_message.lower() for word in ["incomplete", "missing", "needs"]):
        # Need more work - could route back to appropriate agent
        return "synthesizer"  # For simplicity, just finish
    else:
        return "synthesizer"

print("\n✅ Routing Logic Defined:")
print("   🔀 Dynamic routing based on request type")
print("   🔁 Iteration control to prevent infinite loops")
print("   🎯 Quality-based decision making")

# ============================================================================
# PART D: Build the Agent Graph
# ============================================================================

"""
### 🏗️ Part D: Build the LangGraph

Now we assemble everything into a graph structure.
This defines the complete multi-agent workflow.
"""

def create_agent_graph():
    """
    Create and return a LangGraph with all agents and routing.
    """
    print("\n🏗️ Building the Agent Graph...")

    # Initialize the graph with our state schema
    workflow = StateGraph(AgentState)

    # Add nodes (agents)
    workflow.add_node("planner", planner_agent)
    workflow.add_node("researcher", researcher_agent)
    workflow.add_node("coder", coder_agent)
    workflow.add_node("reviewer", reviewer_agent)
    workflow.add_node("synthesizer", synthesizer_agent)

    # Set entry point
    workflow.set_entry_point("planner")

    # Add conditional edges (routing)
    workflow.add_conditional_edges(
        "planner",
        route_after_planner,
        {
            "researcher": "researcher",
            "coder": "coder"
        }
    )

    workflow.add_conditional_edges(
        "researcher",
        route_after_research,
        {
            "coder": "coder",
            "reviewer": "reviewer"
        }
    )

    workflow.add_conditional_edges(
        "coder",
        route_after_coding,
        {
            "reviewer": "reviewer"
        }
    )

    workflow.add_conditional_edges(
        "reviewer",
        route_after_review,
        {
            "synthesizer": "synthesizer"
        }
    )

    # Add edge from synthesizer to END
    workflow.add_edge("synthesizer", END)

    # Compile the graph
    app = workflow.compile()

    print("✅ Graph built successfully!")
    print("\n📊 Graph Structure:")
    print("   START → Planner → [Researcher/Coder] → Reviewer → Synthesizer → END")

    return app

# Create the graph
agent_graph = create_agent_graph()

# ============================================================================
# PART E: Test the Multi-Agent System
# ============================================================================

"""
### 🧪 Part E: Test the System

Let's run our multi-agent system with different types of requests!
"""

def run_agent_system(user_request: str, graph):
    """
    Execute the multi-agent system for a given request.
    """
    print("\n" + "="*60)
    print("🚀 RUNNING MULTI-AGENT SYSTEM")
    print("="*60)
    print(f"\n👤 User Request: {user_request}\n")

    # Initialize state
    initial_state = {
        "user_request": user_request,
        "stage": "start",
        "messages": [],
        "research_data": {},
        "generated_code": "",
        "final_output": "",
        "iteration_count": 0
    }

    # Run the graph
    final_state = graph.invoke(initial_state)

    # Display results
    print("\n" + "="*60)
    print("📊 EXECUTION SUMMARY")
    print("="*60)
    print(f"\n🔢 Total Iterations: {final_state['iteration_count']}")
    print(f"📍 Final Stage: {final_state['stage']}")
    print(f"\n💬 Agent Messages ({len(final_state['messages'])}):")
    for i, msg in enumerate(final_state['messages'], 1):
        print(f"\n{i}. {msg[:150]}...")

    print("\n" + "="*60)
    print("🎯 FINAL OUTPUT")
    print("="*60)
    print(f"\n{final_state['final_output']}")

    if final_state.get('generated_code'):
        print("\n" + "="*60)
        print("💻 GENERATED CODE")
        print("="*60)
        print(f"\n{final_state['generated_code'][:500]}...")

    return final_state

# Test Case 1: Research-focused request
print("\n" + "="*60)
print("TEST CASE 1: Research Request")
print("="*60)

result1 = run_agent_system(
    "What are the key principles of multi-agent systems?",
    agent_graph
)

# Test Case 2: Coding-focused request (uncomment to run)
# print("\n" + "="*60)
# print("TEST CASE 2: Coding Request")
# print("="*60)
#
# result2 = run_agent_system(
#     "Write a Python function to calculate Fibonacci numbers",
#     agent_graph
# )



🕸️ LANGGRAPH MULTI-AGENT SYSTEM

✅ Shared State Schema Defined
   Fields: user_request, stage, messages, research_data,
           generated_code, final_output, iteration_count

✅ Agent Nodes Created:
   🎯 Planner - Analyzes requests and creates plans
   🔍 Researcher - Gathers information
   💻 Coder - Generates code
   ✅ Reviewer - Checks quality
   📝 Synthesizer - Creates final output

✅ Routing Logic Defined:
   🔀 Dynamic routing based on request type
   🔁 Iteration control to prevent infinite loops
   🎯 Quality-based decision making

🏗️ Building the Agent Graph...
✅ Graph built successfully!

📊 Graph Structure:
   START → Planner → [Researcher/Coder] → Reviewer → Synthesizer → END

TEST CASE 1: Research Request

🚀 RUNNING MULTI-AGENT SYSTEM

👤 User Request: What are the key principles of multi-agent systems?


🎯 PLANNER AGENT is analyzing the request...
✅ Plan created: You are a Planning Agent. Analyze this request and create a brief plan:

Request: What are the key p...

🔍 RESEARC

In [38]:

# ============================================================================
# SECTION 11: TEAM PROJECT - Build Your Own Multi-Agent System
# ============================================================================

"""
### 👥 FINAL TASK: Team Project - Software Development Multi-Agent System

**Objective**: Design and implement a multi-agent system that helps with
software development tasks as a team.

**Scenario**: Your team is building a "DevAssist" system - a multi-agent
assistant that helps developers with various tasks:
- Analyzing requirements
- Designing architecture
- Writing code
- Testing code
- Documenting code
- Reviewing and improving code

---

## 📋 Project Guidelines

### Step 1: Form Teams (3-4 students)
- Assign roles: Project Lead, Agent Designer, Integration Specialist, Tester

### Step 2: Design Your Agent System

Answer these questions as a team:

1. **What agents do you need?**
   Example agents:
   - Requirements Analyst Agent
   - Architect Agent
   - Code Generator Agent
   - Test Writer Agent
   - Code Reviewer Agent
   - Documentation Agent

2. **What will each agent do?**
   Define clear responsibilities for each agent.

3. **How will agents communicate?**
   What information needs to be shared? (State design)

4. **What's the workflow?**
   Draw a diagram showing agent interactions.

5. **What tools do agents need?**
   - Code execution?
   - Web search?
   - File access?
   - Testing frameworks?

### Step 3: Implementation Plan

Use this template to implement your system:
"""

# ============================================================================
# TEAM PROJECT TEMPLATE
# ============================================================================

"""
🎯 IMPLEMENTATION TEMPLATE
Copy and customize this for your team project!
"""

# --- YOUR CUSTOM STATE ---
class DevAssistState(TypedDict):
    """
    Define what information flows between your agents.
    Customize this based on your needs!
    """
    # Input
    user_requirement: str

    # Outputs from each agent
    requirements_analysis: str
    architecture_design: str
    generated_code: str
    test_cases: str
    documentation: str
    review_feedback: str

    # Workflow control
    current_stage: str
    iteration_count: int

    # Add more fields as needed!
    # feature_list: list
    # code_quality_score: float
    # etc.

# --- YOUR CUSTOM AGENTS ---

def requirements_analyst_agent(state: DevAssistState) -> DevAssistState:
    """
    TODO: Implement this agent!

    This agent should:
    1. Analyze the user requirement
    2. Break it down into specific features
    3. Identify technical considerations
    4. Update the state with analysis

    Example implementation:
    """
    print("\n📋 Requirements Analyst is working...")

    # Your code here!
    prompt = f"""Analyze this requirement and break it into features:

    Requirement: {state['user_requirement']}

    List 3-5 key features needed:"""

    # Generate response
    if USE_GEMINI:
        response = llm.invoke(prompt)
        analysis = response.content
    else:
        analysis = generate_response(prompt)

    # Update state
    state['requirements_analysis'] = analysis
    state['current_stage'] = 'requirements_done'
    state['iteration_count'] = state.get('iteration_count', 0) + 1

    return state

def architect_agent(state: DevAssistState) -> DevAssistState:
    """
    TODO: Implement this agent!

    This agent should:
    1. Review the requirements analysis
    2. Design system architecture
    3. Choose appropriate patterns
    4. Update state with design
    """
    print("\n🏗️ Architect is designing...")

    # Your implementation here!
    # Use state['requirements_analysis'] as input

    state['architecture_design'] = "TODO: Add architecture design"
    state['current_stage'] = 'architecture_done'
    state['iteration_count'] = state.get('iteration_count', 0) + 1

    return state

def code_generator_agent(state: DevAssistState) -> DevAssistState:
    """
    TODO: Implement this agent!
    """
    print("\n💻 Code Generator is coding...")

    # Your implementation here!

    state['generated_code'] = "TODO: Add generated code"
    state['current_stage'] = 'code_done'
    state['iteration_count'] = state.get('iteration_count', 0) + 1

    return state

def test_writer_agent(state: DevAssistState) -> DevAssistState:
    """
    TODO: Implement this agent!
    """
    print("\n🧪 Test Writer is creating tests...")

    # Your implementation here!

    state['test_cases'] = "TODO: Add test cases"
    state['current_stage'] = 'tests_done'
    state['iteration_count'] = state.get('iteration_count', 0) + 1

    return state

def reviewer_agent(state: DevAssistState) -> DevAssistState:
    """
    TODO: Implement this agent!
    """
    print("\n✅ Reviewer is checking quality...")

    # Your implementation here!

    state['review_feedback'] = "TODO: Add review"
    state['current_stage'] = 'review_done'
    state['iteration_count'] = state.get('iteration_count', 0) + 1

    return state

# --- YOUR ROUTING LOGIC ---

def route_from_requirements(state: DevAssistState) -> str:
    """
    TODO: Implement routing logic!

    Decide which agent should run next after requirements analysis.
    """
    # Your logic here!
    return "architect"  # or "code_generator", etc.

def route_from_architect(state: DevAssistState) -> str:
    """TODO: Implement routing!"""
    return "code_generator"

def route_from_code_generator(state: DevAssistState) -> str:
    """TODO: Implement routing!"""
    return "test_writer"

def route_from_test_writer(state: DevAssistState) -> str:
    """TODO: Implement routing!"""
    return "reviewer"

def route_from_reviewer(state: DevAssistState) -> str:
    """
    TODO: Implement routing!

    Decide if work is complete or needs iteration.
    Consider:
    - Has the iteration limit been reached?
    - Is the quality acceptable?
    - Are there issues that need addressing?
    """
    if state.get('iteration_count', 0) >= 5:
        return "end"

    # Add more sophisticated logic!
    return "end"

# --- BUILD YOUR GRAPH ---

def create_dev_assist_graph():
    """
    TODO: Build your complete agent graph!

    Steps:
    1. Create StateGraph with your custom state
    2. Add all your agent nodes
    3. Set entry point
    4. Add conditional edges with routing
    5. Compile and return
    """
    print("\n🏗️ Building DevAssist Agent Graph...")

    # Initialize graph
    workflow = StateGraph(DevAssistState)

    # TODO: Add your nodes
    workflow.add_node("requirements_analyst", requirements_analyst_agent)
    workflow.add_node("architect", architect_agent)
    workflow.add_node("code_generator", code_generator_agent)
    workflow.add_node("test_writer", test_writer_agent)
    workflow.add_node("reviewer", reviewer_agent)

    # TODO: Set entry point
    workflow.set_entry_point("requirements_analyst")

    # TODO: Add conditional edges
    workflow.add_conditional_edges(
        "requirements_analyst",
        route_from_requirements,
        {
            "architect": "architect",
            "code_generator": "code_generator"
        }
    )

    # Add more edges for your workflow!
    workflow.add_conditional_edges(
        "architect",
        route_from_architect,
        {"code_generator": "code_generator"}
    )

    workflow.add_conditional_edges(
        "code_generator",
        route_from_code_generator,
        {"test_writer": "test_writer"}
    )

    workflow.add_conditional_edges(
        "test_writer",
        route_from_test_writer,
        {"reviewer": "reviewer"}
    )

    workflow.add_conditional_edges(
        "reviewer",
        route_from_reviewer,
        {"end": END}
    )

    # Compile
    app = workflow.compile()

    print("✅ DevAssist Graph Complete!")
    return app

# --- TEST YOUR SYSTEM ---

print("\n" + "="*60)
print("👥 TEAM PROJECT: DevAssist Multi-Agent System")
print("="*60)
print("\n📝 Your task:")
print("   1. Complete the agent implementations above")
print("   2. Design routing logic for your workflow")
print("   3. Test with different software requirements")
print("   4. Evaluate and improve agent behavior")
print("\n🎯 Success Criteria:")
print("   ✅ All agents implemented and working")
print("   ✅ Proper routing between agents")
print("   ✅ System produces useful output")
print("   ✅ Team can explain design decisions")

# Uncomment to test your system:
# dev_assist = create_dev_assist_graph()
#
# test_requirement = "Create a Python web scraper that extracts article titles from a news website"
#
# initial_state = {
#     "user_requirement": test_requirement,
#     "requirements_analysis": "",
#     "architecture_design": "",
#     "generated_code": "",
#     "test_cases": "",
#     "documentation": "",
#     "review_feedback": "",
#     "current_stage": "start",
#     "iteration_count": 0
# }
#
# result = dev_assist.invoke(initial_state)
# print("\n📊 Final Result:")
# print(json.dumps(result, indent=2))

# ============================================================================
# BONUS CHALLENGES
# ============================================================================

"""
### 🌟 Bonus Challenges (Optional)

Once your basic system works, try these enhancements:

1. **Add Persistence**
   - Save agent states to files
   - Load previous sessions
   - Track project history

2. **Implement Error Handling**
   - What if an agent fails?
   - How to retry operations?
   - Graceful degradation

3. **Add Human-in-the-Loop**
   - Allow developers to provide feedback
   - Let humans approve/reject agent decisions
   - Interactive mode

4. **Performance Monitoring**
   - Track agent execution time
   - Measure code quality
   - Collect usage statistics

5. **Advanced Routing**
   - Use LLM to decide routing
   - Dynamic agent selection
   - Adaptive workflows

6. **Tool Integration**
   - Add code execution tools
   - Integrate with GitHub
   - Connect to IDEs
   - Use linters/formatters

7. **Specialized Agents**
   - Security Analyzer Agent
   - Performance Optimizer Agent
   - UI/UX Designer Agent
   - API Designer Agent

---

## 📚 Additional Resources

- LangGraph Documentation: https://langchain-ai.github.io/langgraph/
- CrewAI Documentation: https://docs.crewai.com/
- Multi-Agent Systems Research: https://arxiv.org/abs/2308.08155
- Agent Design Patterns: https://www.anthropic.com/research/building-effective-agents

---

## 🎓 Lab Completion Checklist

- [ ] Loaded and tested an open-source LLM
- [ ] Built a rule-based agent
- [ ] Created an LLM-powered reactive agent
- [ ] Implemented memory in an agent
- [ ] Built a two-agent conversation system
- [ ] Created orchestrator-worker pattern
- [ ] Used tools (search, calculator, Wikipedia)
- [ ] Evaluated agent performance
- [ ] Designed a LangGraph multi-agent system
- [ ] Completed team project with custom agents
- [ ] Tested and debugged the system

---

## 💡 Key Takeaways

1. **Agent Architecture**: Agents combine LLMs with reasoning, memory, and tools
2. **Multi-Agent Patterns**: Different patterns for different problems
   - Orchestrator-Worker: Central coordinator
   - Sequential: Chain of specialists
   - Cyclical: Iterative refinement
3. **State Management**: Critical for agent coordination
4. **Routing Logic**: Enables dynamic, intelligent workflows
5. **Evaluation**: Always measure and improve agent behavior

---

## 🚀 Next Steps

After this lab, you can:
- Build production multi-agent applications
- Experiment with more complex workflows
- Integrate agents into real projects
- Explore advanced agent frameworks
- Research autonomous agent systems

Good luck with your multi-agent systems! 🎉
"""

print("\n" + "="*60)
print("🎓 LAB COMPLETE!")
print("="*60)
print("\n✅ You've learned:")
print("   • How to build individual agents")
print("   • How to coordinate multiple agents")
print("   • How to use LangGraph for complex workflows")
print("   • How to design real-world agent systems")
print("\n🚀 Now build something amazing with your team!")
print("="*60)


👥 TEAM PROJECT: DevAssist Multi-Agent System

📝 Your task:
   1. Complete the agent implementations above
   2. Design routing logic for your workflow
   3. Test with different software requirements
   4. Evaluate and improve agent behavior

🎯 Success Criteria:
   ✅ All agents implemented and working
   ✅ Proper routing between agents
   ✅ System produces useful output
   ✅ Team can explain design decisions

🎓 LAB COMPLETE!

✅ You've learned:
   • How to build individual agents
   • How to coordinate multiple agents
   • How to use LangGraph for complex workflows
   • How to design real-world agent systems

🚀 Now build something amazing with your team!


In [39]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

# This is your shared memory between agents
class DevAssistState(TypedDict, total=False):
    user_requirement: str

    requirements_analysis: str
    architecture_design: str
    generated_code: str
    test_cases: str
    review_feedback: str

    current_stage: str
    iteration_count: int


In [45]:
# ----------------------------------------------------------------------------
# Option 2: Gemini (FASTER, requires API key + correct installation)
# ----------------------------------------------------------------------------
def setup_gemini(api_key: str):
    # Validate API key
    if not api_key or api_key.strip() == "":
        raise ValueError("❌ ERROR: Gemini API key is missing. Set GEMINI_API_KEY first.")

    # Validate dependency
    try:
        from langchain_google_genai import ChatGoogleGenerativeAI
    except ModuleNotFoundError:
        raise ModuleNotFoundError(
            "❌ Missing package: 'langchain-google-genai'\n"
            "Install it using:\n"
            "pip install langchain-google-genai google-generativeai langchain-core"
        )

    os.environ["GOOGLE_API_KEY"] = api_key

    llm = ChatGoogleGenerativeAI(
        model="gemini-2.0-flash",
        temperature=0.7,
        convert_system_message_to_human=True
    )

    print("✅ Gemini API configured successfully!")
    return llm

In [40]:
def requirements_analyst_agent(state: DevAssistState):
    print("\n📋 REQUIREMENTS ANALYST RUNNING...")

    prompt = f"""
You are a senior Requirements Analyst.

Analyze the following requirement:
{state['user_requirement']}

Break it into:
- Key features
- Constraints
- Risks
- Acceptance criteria

Respond in structured sections.
"""

    response = llm.invoke(prompt)["content"]

    state["requirements_analysis"] = response
    state["current_stage"] = "requirements_done"
    state["iteration_count"] = state.get("iteration_count", 0) + 1
    return state


def architect_agent(state: DevAssistState):
    print("\n🏗️ ARCHITECT RUNNING...")

    prompt = f"""
You are a senior software architect.

Based on these requirements:
{state['requirements_analysis']}

Design:
- System architecture
- Components
- Data flow
- APIs (if needed)
- Tech stack
- Security considerations
"""

    response = llm.invoke(prompt)["content"]

    state["architecture_design"] = response
    state["current_stage"] = "architecture_done"
    state["iteration_count"] += 1
    return state


def code_generator_agent(state: DevAssistState):
    print("\n💻 CODE GENERATOR RUNNING...")

    prompt = f"""
You are a senior software engineer.

Based on this architecture:
{state['architecture_design']}

Generate clean, modular, production-level code.
"""

    response = llm.invoke(prompt)["content"]

    state["generated_code"] = response
    state["current_stage"] = "code_done"
    state["iteration_count"] += 1
    return state


def test_writer_agent(state: DevAssistState):
    print("\n🧪 TEST WRITER RUNNING...")

    prompt = f"""
Write detailed test cases based on the following code:

{state['generated_code']}
"""

    response = llm.invoke(prompt)["content"]

    state["test_cases"] = response
    state["current_stage"] = "tests_done"
    state["iteration_count"] += 1
    return state


def reviewer_agent(state: DevAssistState):
    print("\n✅ REVIEWER RUNNING...")

    prompt = f"""
Review the entire output: architecture, code, tests.

Identify:
- bugs
- missing features
- improvements
- security issues
- code smells
"""

    response = llm.invoke(prompt)["content"]

    state["review_feedback"] = response
    state["current_stage"] = "review_done"
    state["iteration_count"] += 1
    return state


In [41]:
def route_from_requirements(state: DevAssistState):
    return "architect"

def route_from_architecture(state: DevAssistState):
    return "code_generator"

def route_from_code(state: DevAssistState):
    return "test_writer"

def route_from_tests(state: DevAssistState):
    return "reviewer"

def route_from_reviewer(state: DevAssistState):
    if state["iteration_count"] >= 5:
        return END
    return END


In [42]:
graph = StateGraph(DevAssistState)

graph.add_node("requirements", requirements_analyst_agent)
graph.add_node("architect", architect_agent)
graph.add_node("code_generator", code_generator_agent)
graph.add_node("test_writer", test_writer_agent)
graph.add_node("reviewer", reviewer_agent)

graph.set_entry_point("requirements")

graph.add_edge("requirements", "architect")
graph.add_edge("architect", "code_generator")
graph.add_edge("code_generator", "test_writer")
graph.add_edge("test_writer", "reviewer")
graph.add_edge("reviewer", END)

workflow = graph.compile()


In [43]:
initial_state = DevAssistState(
    user_requirement="Build me a full-stack login system with OAuth, JWT, and admin panel.",
    iteration_count=0
)

result = workflow.invoke(initial_state)

print("\n================ FINAL OUTPUT ================\n")
for key, value in result.items():
    print(f"\n🔹 {key.upper()}:\n{value}\n")



📋 REQUIREMENTS ANALYST RUNNING...

🏗️ ARCHITECT RUNNING...

💻 CODE GENERATOR RUNNING...

🧪 TEST WRITER RUNNING...

✅ REVIEWER RUNNING...

================ FINAL OUTPUT ================


🔹 USER_REQUIREMENT:
Build me a full-stack login system with OAuth, JWT, and admin panel.


🔹 REQUIREMENTS_ANALYSIS:

You are a senior Requirements Analyst.

Analyze the following requirement:
Build me a full-stack login system with OAuth, JWT, and admin panel.

Break it into:
- Key features
- Constraints
- Risks
- Acceptance criteria

Respond in structured sections.

Key features:
- User registration and authentication
- Password reset functionality
- Social media login integration
- API access for developers
- Admin panel access for administrators

Constraints:
- Minimum 1 month development time
- High priority for security
- Responsive design for mobile devices
- Compatibility with multiple browsers
- Consistent style and design across all pages

Risks:
- High development cost
- Lack of time to comp

In [49]:
# Install missing runtime packages in the notebook environment
!pip install google-generative-ai langchain-google-genai

ERROR: Could not find a version that satisfies the requirement google-generative-ai (from versions: none)
ERROR: No matching distribution found for google-generative-ai


In [5]:
# ============================================================================
# DEVASSIST TEAM AGENT (GEMINI-ONLY VERSION)
# ============================================================================



# Import core libraries
import os
import json
from typing import List, Dict, Any, Optional, TypedDict
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries installed successfully!")

# ----------------------------------------------------------------------------
# Option 2: Gemini (FASTER, requires API key + correct installation)
# ----------------------------------------------------------------------------
def setup_gemini(api_key: str):
    # Validate API key
    if not api_key or api_key.strip() == "":
        raise ValueError("❌ ERROR: Gemini API key is missing. Set GEMINI_API_KEY first.")

    # Validate dependency
    try:
        from langchain_google_genai import ChatGoogleGenerativeAI
    except ModuleNotFoundError:
        raise ModuleNotFoundError(
            "❌ Missing package: 'langchain-google-genai'\n"
            "Install it using:\n"
            "pip install langchain-google-genai google-generativeai langchain-core"
        )

    os.environ["GOOGLE_API_KEY"] = api_key

    llm = ChatGoogleGenerativeAI(
        model="gemini-2.0-flash",
        temperature=0.7,
        convert_system_message_to_human=True
    )

    print("✅ Gemini API configured successfully!")
    return llm

# ============================================================================
# 2. SHARED STATE DEFINITION
# ============================================================================

# To avoid NameError: llm not defined during local runs, provide a lightweight
# fallback LLM implementation and try to initialize a real Gemini LLM if an
# API key is present in the environment.
class _DummyLLM:
    class _Resp:
        def __init__(self, content: str):
            self.content = content

    def invoke(self, prompt: str):
        # Very small heuristic-based canned responses to keep the workflow running.
        # These are intentionally simple and for local/testing only.
        p = prompt.lower()
        if "requirements analyst" in p or "extract" in p and "requirements" in p:
            content = (
                "Requirements:\n"
                "- Req 1: User authentication and authorization\n"
                "- Req 2: CRUD for tasks\n"
                "- Req 3: Role-based dashboard\n\n"
                "Non-functional:\n- Performance: <200ms for common ops\n- Security: hashed passwords, rate limiting\n\n"
                "Risks/Assumptions:\n- External email service availability\n\nSummary: Concise requirements extracted."
            )
        elif "software architect" in p or "design" in p or "architecture" in p:
            content = (
                "Architecture:\n- Architecture style: client-server (REST API)\n"
                "- Components: frontend (React), backend (FastAPI), DB (Postgres), Auth (JWT)\n"
                "- Data flow: client -> API -> DB\n"
                "- Tech stack: Python, FastAPI, React, PostgreSQL\n"
            )
        elif "full-stack" in p or "generate production-quality code" in p or "generate" in p:
            content = (
                "```python\n# Example core backend API (FastAPI)\nfrom fastapi import FastAPI\napp = FastAPI()\n\n@app.get('/')\ndef read_root():\n    return {'msg': 'hello world'}\n```\n"
            )
        elif "qa/test" in p or "unit tests" in p or "pytest" in p:
            content = (
                "```python\n# pytest example\ndef test_sample():\n    assert True\n```\n"
            )
        elif "code reviewer" in p or "review the following" in p:
            content = (
                "Summary: Overall good.\nIssues: Minor missing input validation.\nSuggestions: Add more edge-case tests.\nVerdict: REQUEST CHANGES\n"
            )
        else:
            content = "[DUMMY LLM] Generic fallback response."

        return _DummyLLM._Resp(content)


# Try to initialize real Gemini LLM if possible, otherwise use dummy.
try:
    llm: Any
    GEMINI_KEY = os.environ.get("GEMINI_API_KEY")
    if GEMINI_KEY:
        try:
            llm = setup_gemini(GEMINI_KEY)
        except Exception as e:
            print("⚠️ Could not initialize Gemini LLM, falling back to Dummy LLM. Reason:", e)
            llm = _DummyLLM()
    else:
        # No API key provided — use dummy for local/testing.
        print("ℹ️ GEMINI_API_KEY not set. Using Dummy LLM for local testing.")
        llm = _DummyLLM()
except Exception:
    # Extremely defensive: ensure llm is always defined.
    llm = _DummyLLM()


class DevAssistState(TypedDict, total=False):
    # Input
    user_requirement: str

    # Outputs
    requirements_analysis: str
    architecture_design: str
    generated_code: str
    test_cases: str
    documentation: str
    review_feedback: str

    # Control
    current_stage: str
    iteration_count: int


# ============================================================================
# 3. AGENT IMPLEMENTATIONS
# ============================================================================

def requirements_analyst_agent(state: DevAssistState) -> DevAssistState:
    print("\n📋 Requirements Analyst is working...")

    prompt = f"""
You are a senior Requirements Analyst.

User requirement:
{state['user_requirement']}

Tasks:
1. Extract 3–7 clear functional requirements.
2. Extract non-functional requirements (performance, security, UX, etc.).
3. List main risks and assumptions.
4. Give a short summary.

Format your answer with headings and bullet points.
"""

    response = llm.invoke(prompt)
    analysis = response.content

    state["requirements_analysis"] = analysis
    state["current_stage"] = "requirements_done"
    state["iteration_count"] = state.get("iteration_count", 0) + 1
    return state


def architect_agent(state: DevAssistState) -> DevAssistState:
    print("\n🏗️ Architect is designing...")

    prompt = f"""
You are a senior Software Architect.

Here is the requirements analysis:
{state['requirements_analysis']}

Design:
1. Overall architecture (e.g., client/server, microservices, etc.).
2. Main components/modules and their responsibility.
3. Data flow between components.
4. Tech stack recommendations (backend, frontend, DB, auth).
5. Any important design patterns.

Keep it practical and implementation-ready.
"""

    response = llm.invoke(prompt)
    design = response.content

    state["architecture_design"] = design
    state["current_stage"] = "architecture_done"
    state["iteration_count"] += 1
    return state


def code_generator_agent(state: DevAssistState) -> DevAssistState:
    print("\n💻 Code Generator is coding...")

    prompt = f"""
You are a senior full-stack engineer.

Given this architecture:
{state['architecture_design']}

Generate production-quality code for the core part of the system.
Focus on:
- Clean structure
- Readable functions
- Comments
- No unnecessary boilerplate

Return only code blocks and minimal explanations.
"""

    response = llm.invoke(prompt)
    code = response.content

    state["generated_code"] = code
    state["current_stage"] = "code_done"
    state["iteration_count"] += 1
    return state


def test_writer_agent(state: DevAssistState) -> DevAssistState:
    print("\n🧪 Test Writer is creating tests...")

    prompt = f"""
You are a senior QA/Test engineer.

Given this generated code:
{state['generated_code']}

Write:
1. Unit tests.
2. Integration tests (if relevant).
3. Edge cases.

Use a realistic testing framework (e.g., pytest or Jest depending on language).
Return code blocks for the tests.
"""

    response = llm.invoke(prompt)
    tests = response.content

    state["test_cases"] = tests
    state["current_stage"] = "tests_done"
    state["iteration_count"] += 1
    return state


def reviewer_agent(state: DevAssistState) -> DevAssistState:
    print("\n✅ Reviewer is checking quality...")

    prompt = f"""
You are a senior code reviewer.

Review the following:
- Requirements analysis:
{state['requirements_analysis']}

- Architecture:
{state['architecture_design']}

- Code:
{state['generated_code']}

- Tests:
{state['test_cases']}

Give:
1. A summary of overall quality.
2. Specific issues (bugs, design flaws, missing tests, security risks).
3. Concrete improvement suggestions.
4. A final verdict: "APPROVE" or "REQUEST CHANGES".
"""

    response = llm.invoke(prompt)
    review = response.content

    state["review_feedback"] = review
    state["current_stage"] = "review_done"
    state["iteration_count"] += 1
    return state


# ============================================================================
# 4. ROUTING LOGIC
# ============================================================================

def route_from_requirements(state: DevAssistState) -> str:
    return "architect"

def route_from_architect(state: DevAssistState) -> str:
    return "code_generator"

def route_from_code_generator(state: DevAssistState) -> str:
    return "test_writer"

def route_from_test_writer(state: DevAssistState) -> str:
    return "reviewer"

def route_from_reviewer(state: DevAssistState) -> str:
    # For now: single pass, no iterations
    return END


# ============================================================================
# 5. BUILD WORKFLOW GRAPH (lightweight local implementation)
# ----------------------------------------------------------------------------
# The original code expected a `StateGraph` class from an external library.
# To keep this notebook self-contained and avoid a missing-import error,
# we provide a small, minimal implementation that supports the usage below.
# This simple graph executes nodes sequentially following the single outgoing
# edge from each node until it reaches the END sentinel.
# ============================================================================

END = "END"

class StateGraph:
    def __init__(self, state_type=None):
        self._nodes = {}      # name -> callable(state) -> state
        self._edges = {}      # name -> next_name (or END)
        self._entry = None

    def add_node(self, name: str, func):
        if not callable(func):
            raise ValueError(f"Node function for '{name}' must be callable.")
        self._nodes[name] = func

    def set_entry_point(self, name: str):
        if name not in self._nodes:
            raise KeyError(f"Entry point '{name}' is not a registered node.")
        self._entry = name

    def add_edge(self, src: str, dst: str):
        if src not in self._nodes:
            raise KeyError(f"Source node '{src}' is not registered.")
        # dst may be END or a registered node (we allow dst to be set later)
        self._edges[src] = dst

    def compile(self):
        nodes = self._nodes.copy()
        edges = self._edges.copy()
        entry = self._entry

        class Workflow:
            def __init__(self, nodes, edges, entry):
                self._nodes = nodes
                self._edges = edges
                self._entry = entry

            def invoke(self, state: Dict[str, Any]):
                if self._entry is None:
                    raise RuntimeError("No entry point defined for the workflow.")
                current = self._entry
                # Simple loop guard to prevent infinite loops
                max_steps = 1000
                steps = 0

                while current != END:
                    steps += 1
                    if steps > max_steps:
                        raise RuntimeError("Workflow exceeded maximum steps (possible loop).")
                    if current not in self._nodes:
                        raise KeyError(f"Node '{current}' is not defined in the workflow.")
                    node_fn = self._nodes[current]
                    # Execute node; nodes are expected to return the updated state
                    state = node_fn(state)
                    # Determine next node
                    next_node = self._edges.get(current, END)
                    current = next_node
                return state

        return Workflow(nodes, edges, entry)


# Build the workflow using the lightweight StateGraph above
graph = StateGraph(DevAssistState)

graph.add_node("requirements", requirements_analyst_agent)
graph.add_node("architect", architect_agent)
graph.add_node("code_generator", code_generator_agent)
graph.add_node("test_writer", test_writer_agent)
graph.add_node("reviewer", reviewer_agent)

graph.set_entry_point("requirements")

graph.add_edge("requirements", "architect")
graph.add_edge("architect", "code_generator")
graph.add_edge("code_generator", "test_writer")
graph.add_edge("test_writer", "reviewer")
graph.add_edge("reviewer", END)

devassist_workflow = graph.compile()


# ============================================================================
# 6. FUNCTION YOU CALL FROM YOUR WEBSITE BACKEND
# ============================================================================

def run_devassist(user_requirement: str) -> DevAssistState:
    """Entry point for your website/backend."""
    initial_state: DevAssistState = {
        "user_requirement": user_requirement,
        "iteration_count": 0,
        "current_stage": "start",
    }
    final_state = devassist_workflow.invoke(initial_state)
    return final_state


# ============================================================================
# 7. QUICK LOCAL TEST (REMOVE IN PRODUCTION)
# ============================================================================

if __name__ == "__main__":
    result = run_devassist("Build a full-stack task management web app with login, roles, and dashboard.")
    print("\n==================== FINAL RESULT ====================\n")
    for k, v in result.items():
        print(f"\n🔹 {k}:")
        # Safely print slices for strings, otherwise stringify or pretty-print non-string values
        if isinstance(v, str):
            print(v[:1000])
        else:
            try:
                # Use JSON pretty-print when possible for dicts/lists; fallback to str()
                print(json.dumps(v, indent=2, default=str))
            except Exception:
                print(str(v))
        print()


✅ Libraries installed successfully!
ℹ️ GEMINI_API_KEY not set. Using Dummy LLM for local testing.

📋 Requirements Analyst is working...

🏗️ Architect is designing...

💻 Code Generator is coding...

🧪 Test Writer is creating tests...

✅ Reviewer is checking quality...

==================== FINAL RESULT ====================


🔹 user_requirement:
Build a full-stack task management web app with login, roles, and dashboard.


🔹 iteration_count:
5


🔹 current_stage:
review_done


🔹 requirements_analysis:
Requirements:
- Req 1: User authentication and authorization
- Req 2: CRUD for tasks
- Req 3: Role-based dashboard

Non-functional:
- Performance: <200ms for common ops
- Security: hashed passwords, rate limiting

Risks/Assumptions:
- External email service availability

Summary: Concise requirements extracted.


🔹 architecture_design:
Requirements:
- Req 1: User authentication and authorization
- Req 2: CRUD for tasks
- Req 3: Role-based dashboard

Non-functional:
- Performance: <200ms for 